<a href="https://colab.research.google.com/github/SamTr7/Hackaton/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Fine-tuning rápido YOLO para cacao-agricultura (detección de enfermedad con bounding boxes)



Objetivo del hackatón:

- Entrenar un detector que ubique automáticamente la zona enferma en mazorcas de cacao.

- Clases objetivo para detección: `fito` y `monilia`.

- Imágenes `sana`: se usan como fondo negativo (sin bounding boxes).



Escenario de datos:

- ~100 imágenes RGB con Fito

- ~100 imágenes RGB con Monilia

- ~100 imágenes RGB sanas



En este notebook construimos el esqueleto completo:

1. Preparación de datos YOLO

2. Validación de etiquetas

3. Fine-tuning de modelo preentrenado

4. Evaluación

5. Inferencia y exportación


In [ ]:
# ======================================
# 0) Descarga del dataset en Google Colab
# ======================================
# Ejecuta esta celda primero en Colab.

# Si da error de módulo no encontrado, descomenta:
# !pip install -q dataset-tools

import dataset_tools as dtools

dtools.download(dataset='Cocoa Diseases', dst_dir='~/dataset-ninja/')
print("Descarga solicitada en ~/dataset-ninja/")

# ======================================
# 0.1) Librerías base del proyecto
# ======================================
import random
import shutil
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print("Ultralytics YOLO importado correctamente")

## 1) Configuración del proyecto y rutas



Ajusta estas rutas según cómo tengas guardadas tus imágenes y etiquetas.

- Si ya tienes estructura YOLO (`images/train`, `labels/train`, etc.), usa `USE_PRE_SPLIT_DATA = True`.

- Si tienes todo en carpetas `raw`, usa `USE_PRE_SPLIT_DATA = False` para crear el split automático.


In [ ]:
# ==============================

# Configuración principal

# ==============================

PROJECT_NAME = "cacao_yolo_hackaton"

SEED = 42



# Cambia esta ruta al lugar donde está tu dataset

DATA_ROOT = Path("dataset_cacao")



# Si ya tienes estructura YOLO (images/train, labels/train, ...), usa True

USE_PRE_SPLIT_DATA = True



# Solo se usan cuando USE_PRE_SPLIT_DATA = False

RAW_IMAGES_DIR = DATA_ROOT / "raw_images"

RAW_LABELS_DIR = DATA_ROOT / "raw_labels"



# Carpeta final en formato YOLO

YOLO_DATASET_DIR = DATA_ROOT



# Clases de detección (sana es fondo, no clase)

CLASS_NAMES = ["fito", "monilia"]

NUM_CLASSES = len(CLASS_NAMES)



RUNS_BASE_DIR = Path("runs")



print("Configuración lista:")

print(f"- DATA_ROOT: {DATA_ROOT.resolve()}")

print(f"- YOLO_DATASET_DIR: {YOLO_DATASET_DIR.resolve()}")

print(f"- Clases: {CLASS_NAMES}")


## 2) Estructura esperada del dataset YOLO



Formato recomendado para este proyecto:



```text

dataset_cacao/

  images/

    train/

    val/

    test/

  labels/

    train/

    val/

    test/

```



Cada `.txt` en `labels/*` debe tener líneas tipo YOLO:



`class_id x_center y_center width height`



Todo normalizado entre 0 y 1.



Para imágenes sanas, el archivo `.txt` puede existir vacío (sin cajas).


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}



def list_images(folder: Path):

    if not folder.exists():

        return []

    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS])



def ensure_dirs(base_dir: Path):

    for split in ["train", "val", "test"]:

        (base_dir / "images" / split).mkdir(parents=True, exist_ok=True)

        (base_dir / "labels" / split).mkdir(parents=True, exist_ok=True)



def split_yolo_dataset(raw_images_dir: Path, raw_labels_dir: Path, out_dir: Path,

                       train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9, "Las proporciones deben sumar 1"



    images = list_images(raw_images_dir)

    if len(images) == 0:

        raise FileNotFoundError(f"No se encontraron imágenes en: {raw_images_dir}")



    random.seed(seed)

    random.shuffle(images)



    n = len(images)

    n_train = int(n * train_ratio)

    n_val = int(n * val_ratio)

    n_test = n - n_train - n_val



    splits = {

        "train": images[:n_train],

        "val": images[n_train:n_train + n_val],

        "test": images[n_train + n_val:]

    }



    ensure_dirs(out_dir)



    for split_name, split_imgs in splits.items():

        for img_path in split_imgs:

            dst_img = out_dir / "images" / split_name / img_path.name

            shutil.copy2(img_path, dst_img)



            label_src = raw_labels_dir / f"{img_path.stem}.txt"

            label_dst = out_dir / "labels" / split_name / f"{img_path.stem}.txt"



            if label_src.exists():

                shutil.copy2(label_src, label_dst)

            else:

                # Imagen sana o sin anotación: crear label vacío

                label_dst.write_text("", encoding="utf-8")



    print("Split completado:")

    for split_name, split_imgs in splits.items():

        print(f"- {split_name}: {len(split_imgs)} imágenes")



if not USE_PRE_SPLIT_DATA:

    split_yolo_dataset(

        raw_images_dir=RAW_IMAGES_DIR,

        raw_labels_dir=RAW_LABELS_DIR,

        out_dir=YOLO_DATASET_DIR,

        train_ratio=0.7,

        val_ratio=0.2,

        test_ratio=0.1,

        seed=SEED,

    )

else:

    print("Se asume que el dataset ya está en formato YOLO split.")


## 3) Crear archivo dataset.yaml (YOLO)



YOLO necesita un YAML con rutas y nombres de clases.

Aquí usamos solo dos clases de enfermedad:

- `0: fito`

- `1: monilia`



Las imágenes sanas quedan como fondo (sin clase explícita).


In [ ]:
DATASET_YAML_PATH = YOLO_DATASET_DIR / "dataset.yaml"



dataset_yaml = {

    "path": str(YOLO_DATASET_DIR.resolve()),

    "train": "images/train",

    "val": "images/val",

    "test": "images/test",

    "nc": NUM_CLASSES,

    "names": CLASS_NAMES,

}



with open(DATASET_YAML_PATH, "w", encoding="utf-8") as f:

    yaml.safe_dump(dataset_yaml, f, sort_keys=False, allow_unicode=True)



print(f"dataset.yaml creado en: {DATASET_YAML_PATH.resolve()}")

print(yaml.safe_dump(dataset_yaml, sort_keys=False, allow_unicode=True))


## 4) Validación rápida de anotaciones YOLO



Este paso ayuda a detectar problemas típicos antes de entrenar:

- formato incorrecto en líneas (`5` valores esperados)

- `class_id` fuera de rango

- coordenadas fuera de `[0, 1]`

- cajas con ancho/alto `<= 0`


In [ ]:
def validate_label_file(label_path: Path, num_classes: int):

    errors = []

    text = label_path.read_text(encoding="utf-8").strip()

    if text == "":

        return errors  # archivo vacío es válido para imágenes sanas



    for i, line in enumerate(text.splitlines(), start=1):

        parts = line.split()

        if len(parts) != 5:

            errors.append(f"{label_path.name}: línea {i} -> se esperaban 5 valores, llegaron {len(parts)}")

            continue



        try:

            cls = int(float(parts[0]))

            x, y, w, h = map(float, parts[1:])

        except ValueError:

            errors.append(f"{label_path.name}: línea {i} -> valores no numéricos")

            continue



        if cls < 0 or cls >= num_classes:

            errors.append(f"{label_path.name}: línea {i} -> class_id {cls} fuera de rango [0, {num_classes - 1}]")



        for k, v in zip(["x", "y", "w", "h"], [x, y, w, h]):

            if v < 0 or v > 1:

                errors.append(f"{label_path.name}: línea {i} -> {k}={v:.4f} fuera de [0,1]")



        if w <= 0 or h <= 0:

            errors.append(f"{label_path.name}: línea {i} -> w/h deben ser > 0")



    return errors



def validate_split(split_name: str, root_dir: Path, num_classes: int):

    images_dir = root_dir / "images" / split_name

    labels_dir = root_dir / "labels" / split_name



    if not images_dir.exists() or not labels_dir.exists():

        print(f"[WARN] Split '{split_name}' no encontrado en {root_dir}")

        return



    images = list_images(images_dir)

    label_files = sorted(labels_dir.glob("*.txt"))



    missing_labels = [img for img in images if not (labels_dir / f"{img.stem}.txt").exists()]

    print(f"\nSplit: {split_name}")

    print(f"- Imágenes: {len(images)}")

    print(f"- Labels: {len(label_files)}")

    print(f"- Imágenes sin label .txt: {len(missing_labels)}")



    all_errors = []

    for lf in label_files:

        all_errors.extend(validate_label_file(lf, num_classes))



    if all_errors:

        print(f"- Errores encontrados: {len(all_errors)}")

        print("Primeros 15 errores:")

        for e in all_errors[:15]:

            print("  ", e)

    else:

        print("- Validación OK")



for split in ["train", "val", "test"]:

    validate_split(split, YOLO_DATASET_DIR, NUM_CLASSES)


## 5) Cargar modelo base YOLO (recomendado para 300 imágenes)



Para un dataset pequeño (~300 imágenes), conviene iniciar con un modelo liviano:

- Opción recomendada: `yolov8n.pt` (rápido y estable para fine-tuning)

- Alternativa si tu versión lo soporta: `yolo11n.pt`



Luego, si el tiempo lo permite, puedes probar una variante más grande (`s`).


In [ ]:
MODEL_CHECKPOINT = "yolov8n.pt"  # Cambia a yolo11n.pt si aplica en tu entorno

model = YOLO(MODEL_CHECKPOINT)



print(f"Modelo base cargado: {MODEL_CHECKPOINT}")

model.info()



# Hiperparámetros sugeridos para dataset pequeño (evitar overfitting)

IMG_SIZE = 640

EPOCHS = 80

BATCH = 16 if torch.cuda.is_available() else 8

PATIENCE = 20

DEVICE = 0 if torch.cuda.is_available() else "cpu"



print(f"Device: {DEVICE}")

print(f"EPOCHS={EPOCHS}, BATCH={BATCH}, IMG_SIZE={IMG_SIZE}")


## 6) Fine-tuning del detector



Entrenamos el modelo para detectar lesiones de Fito y Monilia.

Parámetros clave:

- `patience`: corta temprano si no mejora

- `mosaic` moderado: útil, pero controlado en dataset pequeño

- `close_mosaic`: lo desactiva al final para estabilizar cajas


In [ ]:
train_results = model.train(

    data=str(DATASET_YAML_PATH),

    epochs=EPOCHS,

    imgsz=IMG_SIZE,

    batch=BATCH,

    device=DEVICE,

    project=str(RUNS_BASE_DIR),

    name=PROJECT_NAME,

    pretrained=True,

    optimizer="AdamW",

    lr0=1e-3,

    lrf=1e-2,

    weight_decay=5e-4,

    warmup_epochs=3,

    hsv_h=0.015,

    hsv_s=0.5,

    hsv_v=0.3,

    degrees=8.0,

    translate=0.08,

    scale=0.25,

    fliplr=0.5,

    mosaic=0.2,

    mixup=0.0,

    close_mosaic=10,

    patience=PATIENCE,

    seed=SEED,

    deterministic=True,

    val=True,

    plots=True,

)



print("Entrenamiento finalizado")

print(f"Artefactos guardados en: {Path(model.trainer.save_dir).resolve()}")


## 7) Evaluación del modelo



Cargamos el mejor checkpoint y medimos desempeño sobre validación.

Métricas clave para comparar iteraciones:

- `mAP50`

- `mAP50-95`


In [ ]:
best_model_path = Path(model.trainer.best)

best_model = YOLO(str(best_model_path))



metrics = best_model.val(

    data=str(DATASET_YAML_PATH),

    split="val",

    imgsz=IMG_SIZE,

    conf=0.25,

    iou=0.6,

    device=DEVICE,

    plots=True,

)



print(f"Best checkpoint: {best_model_path}")

print(f"mAP50: {metrics.box.map50:.4f}")

print(f"mAP50-95: {metrics.box.map:.4f}")


## 8) Inferencia visual (detección automática de bbox)



Probamos el mejor modelo en imágenes de `test` para ver si localiza bien las zonas enfermas.


In [ ]:
from IPython.display import Image, display



INFER_SOURCE = YOLO_DATASET_DIR / "images" / "test"

PRED_NAME = f"{PROJECT_NAME}_predict"



pred_results = best_model.predict(

    source=str(INFER_SOURCE),

    imgsz=IMG_SIZE,

    conf=0.25,

    iou=0.6,

    device=DEVICE,

    save=True,

    project=str(RUNS_BASE_DIR),

    name=PRED_NAME,

    exist_ok=True,

    max_det=5,

)



pred_dir = RUNS_BASE_DIR / PRED_NAME

print(f"Predicciones guardadas en: {pred_dir.resolve()}")



# Mostrar algunas predicciones

preview = [p for p in sorted(pred_dir.iterdir()) if p.suffix.lower() in IMG_EXTS][:6]

for p in preview:

    display(Image(filename=str(p)))


## 9) Exportar modelo para despliegue



Exportamos a ONNX para integrarlo en backend, edge devices o pipelines de agricultura digital.


In [ ]:
# Si falla por dependencias, instala: !pip install -q onnx onnxsim

onnx_path = best_model.export(format="onnx", opset=12, simplify=True)

print(f"Modelo exportado en: {onnx_path}")


## 10) Recomendaciones rápidas para mejorar en hackatón



- Etiquetado: revisa cajas ajustadas al borde de la lesión (evita cajas demasiado grandes).

- Balance: mantén proporción razonable entre Fito y Monilia en `train` y `val`.

- Calidad visual: agrega fotos con distintas luces, ángulos y estados de madurez.

- Negativos duros: incluye imágenes sanas parecidas a enfermas para reducir falsos positivos.

- Itera rápido: compara 2-3 corridas cambiando `imgsz`, `epochs`, `conf` y augmentations.



Con este esqueleto ya tienes una base sólida para detección automática de enfermedad en cacao.
